# M1: Data Exploration — nuScenes Mini

**Pipeline Position:** Stage 1-2: Ingest to Cloud + Data Quality & Sensor Extraction  
**Input S3 Path:** `s3://av30lab-shared-data-{account_id}/datasets/nuscenes-mini/`  
**Output S3 Path:** `s3://av30lab-user-workspace-{account_id}/users/{profile}/m1/`  
**Source Repo:** [nuScenes devkit](https://github.com/nutonomy/nuscenes-devkit)  
**Instance:** ml.t3.medium (CPU only)

In [ ]:
"""Environment Setup"""
import os
import json
import time
from pathlib import Path

import boto3
import numpy as np
from IPython.display import display, Image as IPImage

# --- S3 Path Configuration ---
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
PROFILE = os.environ.get("USER_PROFILE", "default")

SHARED_BUCKET = os.environ.get("SHARED_BUCKET", f"av30lab-shared-data-{ACCOUNT_ID}")
USER_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}")
NUSCENES_PREFIX = "datasets/nuscenes-mini/"
OUTPUT_PREFIX = f"users/{PROFILE}/m1/"

# --- Clients ---
s3 = boto3.client("s3")
LOCAL_DATA_DIR = Path("/tmp/nuscenes-mini")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Account ID: {ACCOUNT_ID}")
print(f"Profile: {PROFILE}")
print(f"Input: s3://{SHARED_BUCKET}/{NUSCENES_PREFIX}")
print(f"Output: s3://{USER_BUCKET}/{OUTPUT_PREFIX}")
print("\nSetup complete.")

In [ ]:
"""Explore nuScenes-mini structure: download metadata and show all 6 cameras"""
import subprocess

# Download nuScenes-mini metadata + images from S3 (metadata tables AND the
# samples/CAM_*/*.jpg the display below needs).
print("Downloading nuScenes-mini from S3...")
try:
    subprocess.run(
        ["aws", "s3", "sync",
         f"s3://{SHARED_BUCKET}/{NUSCENES_PREFIX}",
         str(LOCAL_DATA_DIR),
         "--quiet"],
        check=True,
    )
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        f"Failed to download the nuScenes-mini dataset from "
        f"s3://{SHARED_BUCKET}/{NUSCENES_PREFIX} (exit {e.returncode}).\n"
        f"  - Is the dataset staged? Admin runs scripts/stage_nuscenes.sh.\n"
        f"  - Does this workspace's execution role have s3:GetObject on the "
        f"shared bucket?"
    ) from e
print(f"Downloaded to: {LOCAL_DATA_DIR}")

# Load nuScenes metadata tables
meta_dir = LOCAL_DATA_DIR / "v1.0-mini"
tables = {}
for json_file in sorted(meta_dir.glob("*.json")):
    table_name = json_file.stem
    with open(json_file) as f:
        tables[table_name] = json.load(f)
    print(f"  {table_name}: {len(tables[table_name])} records")

# List scenes
print(f"\n{'='*60}")
print(f"Scenes in nuScenes-mini: {len(tables.get('scene', []))}")
print(f"{'='*60}")
for scene in tables.get("scene", [])[:10]:
    print(f"  - {scene['name']}: {scene['description']} ({scene['nbr_samples']} samples)")

# nuScenes sample_data has NO "channel" field. Resolve the camera channel
# via join: sample_data.calibrated_sensor_token -> calibrated_sensor.sensor_token
# -> sensor.channel. Built here in cell 2 so cell 4 can reuse cs_to_channel.
_sensor_channel = {s["token"]: s["channel"] for s in tables.get("sensor", [])}
cs_to_channel = {
    cs["token"]: _sensor_channel.get(cs.get("sensor_token"))
    for cs in tables.get("calibrated_sensor", [])
}

def sd_channel(sd):
    """Return the camera/sensor channel for a sample_data record (or None)."""
    return cs_to_channel.get(sd.get("calibrated_sensor_token"))

# Show ONE image per camera for all 6 surround-view cameras, so you can compare
# the vehicle's full 360° view at a glance. nuScenes camera layout:
#   front-left | front | front-right
#    back-left | back  | back-right
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

CAMERAS = [
    "CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT",
    "CAM_BACK_LEFT",  "CAM_BACK",  "CAM_BACK_RIGHT",
]

sample_data = tables.get("sample_data", [])
# First key frame per camera channel (any scene)
first_by_cam = {}
for sd in sample_data:
    ch = sd_channel(sd)
    if ch in CAMERAS and sd.get("is_key_frame") and ch not in first_by_cam:
        first_by_cam[ch] = sd
    if len(first_by_cam) == len(CAMERAS):
        break

print(f"\nCAM_FRONT key frames: "
      f"{sum(1 for sd in sample_data if sd_channel(sd) == 'CAM_FRONT' and sd.get('is_key_frame'))}")

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, cam in zip(axes.ravel(), CAMERAS):
    sd = first_by_cam.get(cam)
    img_path = LOCAL_DATA_DIR / sd["filename"] if sd else None
    if img_path and img_path.exists():
        ax.imshow(mpimg.imread(str(img_path)))
    else:
        ax.text(0.5, 0.5, f"{cam}\n(image not found)",
                ha="center", va="center", transform=ax.transAxes)
    ax.set_title(cam, fontsize=11)
    ax.axis("off")
fig.suptitle("nuScenes-mini — surround-view cameras (one key frame each)", fontsize=13)
plt.tight_layout()
plt.savefig("/tmp/nuscenes_cameras.png", dpi=90, bbox_inches="tight")
plt.show()
display(IPImage(filename="/tmp/nuscenes_cameras.png"))
print("Displayed one key frame from each of the 6 cameras.")

In [ ]:
"""Display sample annotations (bounding boxes, categories)"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

# Load annotations
annotations = tables.get("sample_annotation", [])
categories = {c["token"]: c["name"] for c in tables.get("category", [])}
instances = {i["token"]: i for i in tables.get("instance", [])}

print(f"Total annotations: {len(annotations)}")
print(f"Categories: {len(categories)}")
print()

# Show category distribution
from collections import Counter
cat_counts = Counter()
for ann in annotations:
    inst = instances.get(ann.get("instance_token", ""), {})
    cat_token = inst.get("category_token", "")
    cat_name = categories.get(cat_token, "unknown")
    cat_counts[cat_name] += 1

TOP_N = 15
print(f"Top {TOP_N} annotation categories:")
print("-" * 40)
for cat, count in cat_counts.most_common(TOP_N):
    print(f"  {cat:40s} {count:5d}")

# Plot the SAME Top-15 as a horizontal bar chart, largest at the top.
top_cats = cat_counts.most_common(TOP_N)
fig, ax = plt.subplots(figsize=(12, 7))
labels = [c[0] for c in top_cats][::-1]
values = [c[1] for c in top_cats][::-1]
ax.barh(labels, values, color="steelblue")
for y, v in enumerate(values):
    ax.text(v, y, f" {v}", va="center", fontsize=9)
ax.set_xlabel("Count")
ax.set_title(f"nuScenes-mini: Top {TOP_N} Annotation Categories")
plt.tight_layout()
# Save then display INLINE. (The Agg backend makes plt.show() a no-op, so we
# render the saved PNG explicitly — this is why the chart now appears in the
# notebook instead of only being written to /tmp.)
plt.savefig("/tmp/nuscenes_categories.png", dpi=100, bbox_inches="tight")
plt.show()
display(IPImage(filename="/tmp/nuscenes_categories.png"))
print("\nCategory distribution chart displayed above.")

In [ ]:
"""Write selected samples to user S3 path for downstream modules"""

# Select ALL scenes in nuScenes-mini (10) so downstream modules (M2 captioning,
# M11 pipeline) and your own experiments can draw from the full set. M2 samples
# frames evenly across these scenes (see M2 cell 4), so a larger manifest here
# widens scene diversity without forcing more GPU spend.
selected_scenes = tables.get("scene", [])
scene_tokens = {s["token"] for s in selected_scenes}
scene_name_by_token = {s["token"]: s["name"] for s in selected_scenes}

# Gather sample tokens for selected scenes, and remember each sample's scene so
# every CAM_FRONT frame can be tagged with the scene it came from.
all_samples = tables.get("sample", [])
selected_samples = [s for s in all_samples if s.get("scene_token") in scene_tokens]
selected_sample_tokens = {s["token"] for s in selected_samples}
sample_to_scene = {s["token"]: s.get("scene_token") for s in selected_samples}

# Gather CAM_FRONT key-frame camera data for selected samples
selected_cam_data = [
    sd for sd in tables.get("sample_data", [])
    if sd.get("sample_token") in selected_sample_tokens
    and cs_to_channel.get(sd.get("calibrated_sensor_token")) == "CAM_FRONT"
    and sd.get("is_key_frame")
]

# Per-frame scene NAME, aligned 1:1 with cam_front_files. Lets M2 (and other
# consumers) group/sample frames by scene without re-reading nuScenes metadata.
cam_front_scenes = [
    scene_name_by_token.get(sample_to_scene.get(sd.get("sample_token")), "unknown")
    for sd in selected_cam_data
]

# Build output manifest
output_manifest = {
    "scenes": [s["name"] for s in selected_scenes],
    "num_samples": len(selected_samples),
    "num_cam_front_frames": len(selected_cam_data),
    "cam_front_files": [sd["filename"] for sd in selected_cam_data],
    "cam_front_scenes": cam_front_scenes,
    "source_bucket": SHARED_BUCKET,
    "source_prefix": NUSCENES_PREFIX,
}

# Upload manifest to user bucket
manifest_key = f"{OUTPUT_PREFIX}manifest.json"
s3.put_object(
    Bucket=USER_BUCKET,
    Key=manifest_key,
    Body=json.dumps(output_manifest, indent=2),
    ContentType="application/json"
)
print(f"Uploaded manifest: s3://{USER_BUCKET}/{manifest_key}")

# Copy selected scene metadata
scene_meta_key = f"{OUTPUT_PREFIX}selected_scenes.json"
s3.put_object(
    Bucket=USER_BUCKET,
    Key=scene_meta_key,
    Body=json.dumps(selected_scenes, indent=2),
    ContentType="application/json"
)
print(f"Uploaded scene metadata: s3://{USER_BUCKET}/{scene_meta_key}")

# Per-scene frame breakdown (confirms all 10 scenes are represented)
from collections import Counter
_per_scene = Counter(cam_front_scenes)
print(f"\nSelected {len(selected_scenes)} scenes with {len(selected_cam_data)} CAM_FRONT frames")
print("Frames per scene:")
for _name in [s["name"] for s in selected_scenes]:
    print(f"  {_name}: {_per_scene.get(_name, 0)}")
print(f"Output ready for M2 at: s3://{USER_BUCKET}/{OUTPUT_PREFIX}")

In [ ]:
"""Cost Analysis"""
# ml.t3.medium pricing: $0.056/hr (us-west-2)
INSTANCE_COST_PER_HOUR = 0.056  # USD
KRW_RATE = 1370  # USD to KRW

# Estimated execution time for this notebook
estimated_minutes = 15
estimated_hours = estimated_minutes / 60

cost_usd = INSTANCE_COST_PER_HOUR * estimated_hours
cost_krw = cost_usd * KRW_RATE

print("=" * 50)
print("M1 Data Exploration — Cost Analysis")
print("=" * 50)
print(f"Instance type:     ml.t3.medium")
print(f"Instance cost:     ${INSTANCE_COST_PER_HOUR:.3f}/hr")
print(f"Estimated time:    {estimated_minutes} minutes")
print(f"Compute cost:      ${cost_usd:.4f} USD ({cost_krw:.0f} KRW)")
print(f"S3 storage cost:   ~$0.001 USD (negligible)")
print(f"Total estimated:   ${cost_usd:.4f} USD ({cost_krw:.0f} KRW)")
print("=" * 50)
print("\nNote: CPU-only module — minimal cost.")

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m01-data-exploration")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")